In [15]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
from SDRUtils.products._swaptions.pricer import (
    usd_swaption_straddle_pricer_from_row,
    usd_swaption_leg_pricer_from_row,
    usd_swaption_dealer_risk_reversal_skew_from_row,
    USDSwaptionStraddlePricerResult,
    USDSwaptionLegPricerResult,
	USDSwaptionDealerRiskReversalSkewResult,
    _compute_swaption_leg_greeks,
    USDSwaptionVerticalSpreadPricerResult,
    usd_swaption_vertical_spread_pricer_from_row
)

In [17]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 1, 21)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

# from SDRUtils.data.builder import SDRDataBuilder
# sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
# df = sdr.grab_sdr_trades(
# 	start_timestamp=start,
# 	end_timestamp=end,
# 	agency="CFTC",
# 	asset_class="RATES",
# )
# df

FETCHING ERIS INTRADAY DISC CURVE...: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


In [18]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True, merge_package_legs=True)
# sdf

PRICING VERTICAL SPREADS...:   0%|          | 0/17 [00:00<?, ?it/s]WARNING	Task(Task-2) SDRUtils.packages.swaption_packages:swaption_packages.py:_price_vertical_spreads()- Failed to price vertical spread package VERTICAL_SPREAD_1x1_4d9bdb77993c: root not bracketed: f[0,1] -> [1.129317e+07,3.272647e+07]
WARNING	Task(Task-2) SDRUtils.packages.swaption_packages:swaption_packages.py:_price_vertical_spreads()- Failed to price vertical spread package VERTICAL_SPREAD_1x1_75b4dbf02f1e: root not bracketed: f[0,1] -> [4.146274e+06,5.107882e+08]
PRICING OUTRIGHTS...: 100%|██████████| 211/211 [00:02<00:00, 95.72it/s] 


In [19]:
sdf[sdf["rr_atmf"].notna()]
# .head(1).to_dict(orient="records")
# [
# 	["trade_id", "execution_timestamp", "trade_label", "notional", "strike", "premium", "platform_identifier", "exercise_style"]
# ]
# .head(3).to_dict(orient="records")
# sdf[sdf["ladder_notionals"].notna()].head(3)[
# 	
# ]

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,outright_atmf,outright_strike_offset_bps,outright_strike_offset_rounded_bps,outright_moneyness,outright_bpvol_yr,outright_fwd_premium,outright_dv01,outright_vega01,outright_gamma01,outright_theta1d
233,NEWT-TRAD,1802922016000000501 / 1802929775000000301 / 18...,2026-01-21T14:29:41+00:00 / 2026-01-21T14:30:4...,2026-01-21,2027-01-21,SWAPTION_PAYER / SWAPTION_RECEIVER / SWAPTION_...,USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER...,1e+08 / 1e+08 / 1.8e+07 / 1.8e+07,USD,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,NEWT-TRAD,1805839456000000101 / 1805839759000000101 / 18...,2026-01-21T20:54:06+00:00 / 2026-01-21T20:54:3...,2026-01-21,2031-01-21,SWAPTION_PAYER / SWAPTION_RECEIVER / SWAPTION_...,USD-SOFR-OIS Compound 1D CONSTANT 5Yx10Y PAYER...,1.5e+08 / 1.5e+08 / 7.7e+07 / 7.7e+07,USD,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [73]:
import ujson as json


def format_swaption_pricing_results(
    results: USDSwaptionStraddlePricerResult | USDSwaptionLegPricerResult | USDSwaptionDealerRiskReversalSkewResult,
):
    if isinstance(results, USDSwaptionDealerRiskReversalSkewResult):
        output = {
            "trade": results.trade_label,
            "atm_strike": results.atm_strike * 100,
            "otm_payer_strike": results.otm_payer_strike * 100,
            "otm_receiver_strike": results.otm_receiver_strike * 100,
            "wing_strike_width": results.wing_strike_width,
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_payer_bpvol": results.otm_payer_bpvol_yr,
            "otm_receiver_bpvol": results.otm_receiver_bpvol_yr,
            "payer_skew_bpvol_yr": results.payer_skew_bpvol_yr,
            "receiver_skew_bpvol_yr": results.receiver_skew_bpvol_yr,
            "skew_bpvol": results.skew_bpvol_yr,
            "atm_notional": results.atm_notional,
            "wing_notional": results.wing_notional,
            "otm_payer_vega01": results.otm_payer_vega01,
            "otm_receiver_vega01": results.otm_receiver_vega01,
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
            "wing_dv01": results.wing_dv01
        }
    elif isinstance(results, USDSwaptionVerticalSpreadPricerResult):
        output = {
            "trade": results.trade_label,
            "spread_type": results.spread_type,
            # Strikes
            "atm_strike": results.atm_strike * 100,
            "otm_strike": results.otm_strike * 100,
            "strike_width_bps": results.strike_width_bps,
            "atm_strike_offset": results.atm_strike_offset,
            "otm_strike_offset": results.otm_strike_offset,
            # Vols
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_bpvol": results.otm_bpvol_yr,
            "vol_spread_bpvol": results.vol_spread_bpvol_yr,
            # Notionals
            "atm_notional": results.atm_notional,
            "otm_notional": results.otm_notional,
            "notional_ratio": results.notional_ratio,
            # Premiums
            "net_premium": results.net_premium,
            "atm_premium": results.atm_premium,
            "otm_premium": results.otm_premium,
            # ATM leg Greeks
            "atm_dv01": results.atm_dv01,
            "atm_gamma01": results.atm_gamma01,
            "atm_vega01": results.atm_vega01,
            
            "atm_theta1d": results.atm_theta1d,
            # OTM leg Greeks
            "otm_dv01": results.otm_dv01,
            "otm_gamma01": results.otm_gamma01,
            "otm_vega01": results.otm_vega01,
            "otm_theta1d": results.otm_theta1d,
            # Aggregate Greeks
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }
    else:
        output = {
            "trade": results.trade_label,
            "prem": (results.fwd_prem / results.notional) * 10_000,
            "bpvol": results.bpvol_yr,
            "bpvol_day": results.bpvol_yr / np.sqrt(252),
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }

    print(json.dumps(output, indent=4))

In [75]:
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[168], pricer))
format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[490], pricer))

{
    "trade": "USD-SOFR-OIS Compound 1D CONSTANT 1Yx10Y PAYER EURO VANILLA PHYS",
    "prem": 480.0,
    "bpvol": 72.25211145222885,
    "bpvol_day": 4.5514552048076045,
    "dv01": 852.9995830946937,
    "gamma01": 663.3399956226453,
    "vega01": 64548.32454711436,
    "theta1d": -6340.930749212392
}


In [18]:
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[199], pricer))
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[96], pricer))

# format_swaption_pricing_results(usd_swaption_dealer_risk_reversal_skew_from_row(risk_reversal_row=sdf.loc[286], pricer=pricer))

In [53]:

row = sdf.loc[168]

display(row.to_dict())
_compute_swaption_leg_greeks(
	pricer,
	row["expiration_date"],
	row["underlying_expiration_date"],
	row["strike"],
	row["notional"],
	row["premium"],
	"receiver" if "rec" in row["product_type"].lower() else "payer",
)

{'event_action': 'NEWT-TRAD',
 'trade_id': '1745734985000000401',
 'execution_timestamp': Timestamp('2026-01-15 20:56:16+0000', tz='UTC'),
 'effective_date': Timestamp('2026-01-15 00:00:00'),
 'expiration_date': Timestamp('2028-01-10 00:00:00'),
 'product_type': 'SWAPTION_PAYER',
 'trade_label': 'USD-SOFR-OIS Compound 1Y CONSTANT 2Yx30Y PAYER EURO VANILLA PHYS',
 'notional': 100000000.0,
 'notional_currency': 'USD',
 'is_notional_capped': False,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2058-01-12 00:00:00'),
 'tenor_years': 30.027397260273972,
 'tenor_label': '30Y',
 'forward_start_years': 1.9863013698630136,
 'forward_label': '2Y',
 'premium': 450000.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.05234,
 'upi_underlier_name': 'NA/Swap OIS USD',
 'unique_product_identifier': 'QZZLNQ2D4JQT',
 'platform_identifier': 'BILT',
 'cleared': 'N',
 'package_indicator': False,
 'package_transaction_price': '',
 'option_pre

_SwaptionLegGreeks(bpvol_yr=50.80008511489668, dv01=13146.336552531715, gamma01=394.24023436883004, vega01=34351.294088425864, theta1d=1203.1005319558317, strike_offset=100)

In [452]:
# temp = sdf.loc[422].copy()

# temp["premium"] = temp["premium"] / 4
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(temp, pricer))
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(temp, pricer))

In [123]:
# ids = [1712299926000000501, 1712299925000000401]

ids = [
1745547082000000201,
1745547081000000101


]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("_temp_raw_raw_trades.csv",index=False)

df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df[df["Original Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'Dissemination Identifier': '1745547082000000201',
  'Original Dissemination Identifier': '',
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Amendment indicator': None,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'N',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-01-15 00:00:00'),
  'Expiration Date': Timestamp('2026-02-17 00:00:00'),
  'Maturity date of the underlier': datetime.date(2056, 2, 19),
  'Non-standardized term indicator': False,
  'Platform identifier': 'BILT',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': False,
  'Notional amount-Leg 1': '60,000,000',
  'Notional amount-Leg 2': '60,000,000',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notional q

In [181]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

start = NY_tz.localize(datetime.datetime(2026, 1, 1, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 1, 16, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 11/11 [00:00<00:00, 37.16it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,1605354285000000101,,NEWT,TRAD,2026-01-02 00:01:01+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZ3547LN59PD,NA/Swap Flt Flt OIS AUD,AUD-AONIA-OIS-COMPOUND vs AUD-BBR-BBSW
1,1605354387000000101,1605354285000000101,CORR,,2026-01-02 00:01:01+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZ3547LN59PD,NA/Swap Flt Flt OIS AUD,AUD-AONIA-OIS-COMPOUND vs AUD-BBR-BBSW
2,1605354388000000201,1605354285000000101,CORR,,2026-01-02 00:01:04+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZ3547LN59PD,NA/Swap Flt Flt OIS AUD,AUD-AONIA-OIS-COMPOUND vs AUD-BBR-BBSW
3,1605354389000000301,1605354285000000101,TERM,NOVA,2026-01-02 00:01:47+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZ3547LN59PD,NA/Swap Flt Flt OIS AUD,AUD-AONIA-OIS-COMPOUND vs AUD-BBR-BBSW
4,1605394595000000101,1319974356,MODI,TRAD,2026-01-02 00:10:02+00:00,False,IR,None,N,False,...,EUR,1.0,,,NaN,None,None,QZVDD5NT5G65,NA/O Call Epn Fxd Flt EUR,NA/Swap Fxd Flt EUR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288953,1762554444000000101,1745587218000001001,CORR,,2026-01-16 23:06:03+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZZGWPNBF5R3,NA/O Call Epn OIS USD,NA/Swap OIS USD
288954,1762555069000000101,1745587219000001101,CORR,,2026-01-16 23:06:34+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZZGWPNBF5R3,NA/O Call Epn OIS USD,NA/Swap OIS USD
288955,1762556426000000101,1745587212000000401,CORR,,2026-01-16 23:07:17+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZNLQ8T0N0SX,NA/O P Epn OIS USD,NA/Swap OIS USD
288956,1762556820000000101,1745587213000000501,CORR,,2026-01-16 23:07:45+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZNLQ8T0N0SX,NA/O P Epn OIS USD,NA/Swap OIS USD


In [188]:
# upis = pd.read_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Option-Non_Standard.csv")
upis = pd.read_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Option-CapFloor.csv")
# upis = pd.read_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Option-Debt_Option.csv")
# upis = pd.read_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Credit-Option-Index_Swaption.csv")
# upis = pd.read_csv(r'C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Rates-Forward-Debt.csv')
df[df["Unique Product Identifier"].isin(upis["Identifier_UPI"])]["Unique Product Identifier"].value_counts()

Unique Product Identifier
QZQJWDQ4V0VJ    249
QZXNP136XML0    226
QZHM38VZ056X     79
QZ7VKKVLNLP0     78
QZXGTP6KDS9R     63
               ... 
QZ5GLBK5D5FB      1
QZXV9CC363RC      1
QZQV209FV1PB      1
QZQXKFBXK1M5      1
QZD06FDBL5TF      1
Name: count, Length: 106, dtype: int64

In [ ]:
cols = [
    'Event timestamp',
	'Effective Date',
	'Expiration Date',
	'Maturity date of the underlier',
	'Notional amount-Leg 1',
	# 'Notional amount-Leg 2',
	'Option Premium Amount',
    'First exercise date',
	'Fixed rate-Leg 1',
	'Strike Price',
    # "Platform identifier",
    "Unique Product Identifier"
]
# .to_csv("unknown_exotics_trades.csv")

# df[df["Unique Product Identifier"].isin(upis["Identifier_UPI"])]["Platform identifier"].value_counts()
# df[(df["Platform identifier"] == "ISWV") & (df["Unique Product Identifier"].isin(upis["Identifier_UPI"]))][cols]["Unique Product Identifier"].value_counts()
df[(df["Unique Product Identifier"].isin(["QZG25MC1NLGS", "QZ1D6VBF58LM"]))][cols]

,Event timestamp,Effective Date,Expiration Date,Maturity date of the underlier,Notional amount-Leg 1,Option Premium Amount,First exercise date,Fixed rate-Leg 1,Strike Price,Unique Product Identifier


In [113]:
0.32  * 15.87

5.0784

In [114]:
38.106 * 2

76.212

In [ ]:
from SDRUtils.anna_dsb_upis.utils import AnnaDSBFetcher


ab = AnnaDSBFetcher()

updated_upi_product_df = ab.get_anna_dsb_upis(
	asset_class="Credit",
	instrument_type="Option",
	product="Index_Swaption",
	token="eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6ImN4WExKNEZ5UVAxdnl0dEtGX1g5dCJ9.eyJodHRwczovL3Byb2QuYW5uYS1kc2IuY29tL3VzZXJuYW1lIjoiamlib2JhYjIyMkBlbGFmYW5zLmNvbSIsImh0dHBzOi8vcHJvZC5hbm5hLWRzYi5jb20vZmRsVDBBY2Nlc3NVUEkiOmZhbHNlLCJodHRwczovL3Byb2QuYW5uYS1kc2IuY29tL3NlYXJjaExpbWl0VVBJIjo1LCJodHRwczovL3Byb2QuYW5uYS1kc2IuY29tL2dyb3VwSWRzIjpbIjIwLjEyMDAuMS9VUElfUmVhZCJdLCJpc3MiOiJodHRwczovL2F1dGguYW5uYS1kc2IuY29tLyIsInN1YiI6ImF1dGgwfDY5NmVhOWVkYWIzNmM2MDRhNjhhYmIzMSIsImF1ZCI6WyJndWkiLCJodHRwczovL2NmLWFubmEtZHNiLmV1LmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3Njg4NjEyMTMsImV4cCI6MTc2ODg2MjExMywic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBlbWFpbCBvZmZsaW5lX2FjY2VzcyIsImF6cCI6Ikg3SEVMdVhBakZMSFJ0NW5aanJhMklZdmZKYzRHb1NFIn0.Yi47od4t-odIIRdxmJYMeTbYINSEdhlx0jXD3zkCFtBlq6BQlEKQgrpWTT7_BW3UMGdE6LCHo07F2T0bnFP3sbcCw6JqCWlk4zeI3o1KUfwIGZovrMfqnid9y_uu8SjSHWjxT3cqx20mQP9kjE4KBEFUJAJxSpTE1w0ARHi3nhVTHWfVTVFb0nGlT76EQNxbrTXG-iD_SN0jvzSstUVXOTiUf_K0VWa5lydLgnXNiXGjuz4EHvS6vdNIHxHGKQuX-c6oCB2BUq6IdcWxybl2M0pKuLtdp7nk4DLXS5HATsv5dRnO3UAFVv05CKzfaVZ5pf7guBS0LqPFoupLDaelRQ",
	num_of_iterations=6493 * 5,
	max_concurrent_tasks=256,
	max_keepalive_connections=12,
)
updated_upi_product_df.to_csv(r"C:\Users\chris\clee\ARBS\SDRUtils\anna_dsb_upis\Credit-Option-Index_Swaption.csv")